# 🚀 Experimento 512 tokens — RoBERTa clínico (Google Colab)

**Línea experimental aparte.** Reentrena el mejor modelo (auditor RoBERTa clínico, 7 clases) usando `max_length = 512` en lugar de 256, para ver si más contexto mejora el aprendizaje. Todo lo demás se mantiene idéntico al baseline de 256 (mismo split semilla 42, misma augmentación solo-train, mismo LR, mismas épocas), de modo que la única variable que cambia es la longitud de tokens.

**Baselines a superar (entrenados con 256 tokens):**
- F1-macro (7 clases, test real): **0,5871**
- AUC zona (riesgo vs. seguro): **0,93**

---

### Instrucciones (leer antes de correr)

1. **Activar GPU:** menú `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → Acelerador por hardware: **GPU** (T4 sirve).
2. **Subir el dataset:** la celda 3 te pedirá subir `dataset_clean.csv` (el mismo que generaste localmente, está en `data/processed/`).
3. Ejecutar las celdas en orden (`Entorno de ejecución` → `Ejecutar todo`).
4. Al final, comparar el F1-macro y el AUC contra los baselines de arriba.

⚠️ **Expectativa honesta:** la entrada del auditor son **solo las observaciones**, que suelen ser cortas. La celda 4 mide cuántos informes superan realmente los 256 tokens; si son muy pocos, el techo de mejora con 512 es pequeño. Aun así el experimento es limpio: elimina por completo la truncación y permite ver si los informes largos (que a 256 se cortan) contienen señal de riesgo.

## 1 · Instalar dependencias y verificar GPU

In [1]:
!pip install -q transformers
import torch
print("PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No hay GPU. Activa GPU en: Entorno de ejecución -> Cambiar tipo de entorno de ejecución.")

PyTorch: 2.11.0+cu128
GPU disponible: True
GPU: Tesla T4


## 2 · Librerías y configuración

In [2]:
import numpy as np, pandas as pd, random
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, roc_auc_score, confusion_matrix

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'

# >>> La única variable que cambia respecto al baseline <<<
MAX_LENGTH = 512     # baseline = 256. Probar 512 (o 384, que ya cubre el máximo del corpus).

MODELO='PlanTL-GOB-ES/roberta-base-biomedical-clinical-es'
MODELO_ALT='BSC-LT/roberta-base-biomedical-clinical-es'
LR=3e-5; EPOCHS=15; PATIENCE=4; BATCH=8   # batch 8 para que 512 tokens quepa en T4
print(f"Dispositivo: {DEVICE} | max_length = {MAX_LENGTH} | batch = {BATCH}")

Dispositivo: cuda | max_length = 512 | batch = 8


## 3 · Subir el dataset curado

Se abrirá un selector de archivos. Sube `dataset_clean.csv`.

In [3]:
from google.colab import files
subidos = files.upload()   # selecciona dataset_clean.csv
nombre = list(subidos.keys())[0]
df = pd.read_csv(nombre, encoding='utf-8')
print(f"Cargado: {nombre} | {len(df)} filas")
assert 'auditor_input' in df.columns and 'BI-RADS' in df.columns, "Faltan columnas auditor_input / BI-RADS"

# Alternativa con Google Drive (descomenta si prefieres):
# from google.colab import drive; drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/ruta/dataset_clean.csv')

Saving dataset_clean.csv to dataset_clean.csv
Cargado: dataset_clean.csv | 4357 filas


## 4 · Diagnóstico: ¿cuántos informes superan 256 tokens?

Mide el techo real de mejora. Si casi ninguno supera 256, 512 cambiará poco.

In [4]:
try: tok_diag=AutoTokenizer.from_pretrained(MODELO)
except Exception: MODELO=MODELO_ALT; tok_diag=AutoTokenizer.from_pretrained(MODELO)
longs=df['auditor_input'].astype(str).apply(lambda t: len(tok_diag(t,truncation=False)['input_ids']))
print(f"Tokens en la entrada del auditor (solo observaciones):")
print(f"  mediana={int(longs.median())} | p95={int(longs.quantile(.95))} | máx={int(longs.max())}")
for umbral in [256,384,512]:
    n=int((longs>umbral).sum()); print(f"  Informes que superan {umbral} tokens: {n} ({100*n/len(longs):.1f}%)")
print("\nSi el % que supera 256 es bajo, el efecto esperable de pasar a 512 es pequeño.")

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/540k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Tokens en la entrada del auditor (solo observaciones):
  mediana=97 | p95=154 | máx=238
  Informes que superan 256 tokens: 0 (0.0%)
  Informes que superan 384 tokens: 0 (0.0%)
  Informes que superan 512 tokens: 0 (0.0%)

Si el % que supera 256 es bajo, el efecto esperable de pasar a 512 es pequeño.


## 5 · División y augmentación (idénticas al baseline)

In [5]:
X=df['auditor_input'].values; y=df['BI-RADS'].values
X_tv,X_test,y_tv,y_test=train_test_split(X,y,test_size=0.15,random_state=SEED,stratify=y)
X_train,X_val,y_train,y_val=train_test_split(X_tv,y_tv,test_size=0.176,random_state=SEED,stratify=y_tv)

clases=np.unique(y_train)
pesos=compute_class_weight('balanced',classes=clases,y=y_train)
peso_vec=np.ones(7,dtype=np.float32)
for c,w in zip(clases,pesos): peso_vec[c]=w
peso_tensor=torch.tensor(peso_vec,dtype=torch.float32).to(DEVICE)

def aumentar_texto(texto):
    variaciones=[('bordes irregulares','márgenes irregulares'),('bordes espiculados','márgenes espiculados'),
        ('bordes mal definidos','límites imprecisos'),('imagen nodular','nódulo'),('nódulo','imagen nodular'),
        ('lesión nodular','nódulo'),('heterogéneamente densas','de densidad heterogénea'),
        ('muy densas','extremadamente densas'),('calcificaciones sospechosas','depósitos cálcicos sospechosos'),
        ('microcalcificaciones','calcificaciones puntiformes agrupadas'),('mama derecha','hemimama derecha'),
        ('mama izquierda','hemimama izquierda'),('se visualiza','se observa'),('se observa','se visualiza'),
        ('se evidencia','se observa')]
    t=texto
    for o,r in random.sample(variaciones,min(3,len(variaciones))):
        if o in t: t=t.replace(o,r,1)
    return t

mask=np.isin(y_train,[3,4,5,6]); Xm,ym=X_train[mask],y_train[mask]
ta,la=[],[]
for txt,lab in zip(Xm,ym):
    for _ in range(3): ta.append(aumentar_texto(txt)); la.append(lab)
X_train_aug=np.concatenate([X_train,np.array(ta)]); y_train_aug=np.concatenate([y_train,np.array(la)])
idx=np.random.permutation(len(X_train_aug)); X_train_aug,y_train_aug=X_train_aug[idx],y_train_aug[idx]
print(f"Train {len(X_train)} -> aumentado {len(X_train_aug)} | Val {len(X_val)} | Test {len(X_test)} (riesgo: {(y_test>=4).sum()})")

Train 3051 -> aumentado 3387 | Val 652 | Test 654 (riesgo: 11)


## 6 · Dataset, modelo y entrenamiento (con precisión mixta para T4)

In [6]:
tokenizer=AutoTokenizer.from_pretrained(MODELO)
class DS(Dataset):
    def __init__(self,t,l): self.t=list(t); self.l=list(l)
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        e=tokenizer(str(self.t[i]),truncation=True,padding='max_length',max_length=MAX_LENGTH,
                    return_tensors='pt',return_token_type_ids=False)
        return {'input_ids':e['input_ids'].squeeze(0),'attention_mask':e['attention_mask'].squeeze(0),
                'labels':torch.tensor(self.l[i],dtype=torch.long)}
tl=DataLoader(DS(X_train_aug,y_train_aug),batch_size=BATCH,shuffle=True)
vl=DataLoader(DS(X_val,y_val),batch_size=BATCH)
tt=DataLoader(DS(X_test,y_test),batch_size=BATCH)

class Auditor(nn.Module):
    def __init__(self,m,n=7,dp=0.5):
        super().__init__(); self.enc=AutoModel.from_pretrained(m)
        self.dp=nn.Dropout(dp); self.cl=nn.Linear(self.enc.config.hidden_size,n)
    def forward(self,ids,mask):
        return self.cl(self.dp(self.enc(input_ids=ids,attention_mask=mask).last_hidden_state[:,0,:]))
model=Auditor(MODELO).to(DEVICE)

crit=nn.CrossEntropyLoss(weight=peso_tensor)
opt=torch.optim.AdamW(model.parameters(),lr=LR)
sch=get_linear_schedule_with_warmup(opt,0,len(tl)*EPOCHS)
usar_amp=(DEVICE=='cuda'); scaler=torch.cuda.amp.GradScaler(enabled=usar_amp)

def epoca(loader,train=True):
    model.train() if train else model.eval()
    tot=0.0; P=[]; R=[]
    with torch.set_grad_enabled(train):
        for j,b in enumerate(loader):
            ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE); lab=b['labels'].to(DEVICE)
            if train: opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=usar_amp):
                logits=model(ids,mask); loss=crit(logits,lab)
            if train:
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
            tot+=loss.item(); P.extend(logits.argmax(1).cpu().numpy()); R.extend(lab.cpu().numpy())
            if train and (j+1)%100==0: print(f"   batch {j+1}/{len(loader)}",flush=True)
    return tot/len(loader), f1_score(R,P,average='macro',zero_division=0)

print(f"🏋️ Entrenando RoBERTa clínico con max_length={MAX_LENGTH}")
print("Época | TrainLoss | ValLoss | TrainF1 | ValF1")
mejor,sin=0.0,0
for ep in range(1,EPOCHS+1):
    trl,trf=epoca(tl,True); vll,vlf=epoca(vl,False); m=""
    if vlf>mejor: mejor,sin=vlf,0; torch.save(model.state_dict(),'mejor_512.pt'); m="  <- mejor"
    else: sin+=1
    print(f"  {ep:2d}  |  {trl:.4f}  |  {vll:.4f}  |  {trf:.4f}  |  {vlf:.4f}{m}",flush=True)
    if sin>=PATIENCE: print(f"Early stopping en época {ep}"); break
print(f"🏆 Mejor Val F1 = {mejor:.4f}")

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_1658/50223184.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  usar_amp=(DEVICE=='cuda'); sc

🏋️ Entrenando RoBERTa clínico con max_length=512
Época | TrainLoss | ValLoss | TrainF1 | ValF1


model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

...

   7  |  0.1259  |  1.1316  |  0.9607  |  0.4629


/tmp/ipykernel_1658/50223184.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=usar_amp):


   batch 100/424
   batch 200/424
   batch 300/424
   batch 400/424


/tmp/ipykernel_1658/50223184.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=usar_amp):


   8  |  0.1123  |  1.0531  |  0.9606  |  0.4703
Early stopping en época 8
🏆 Mejor Val F1 = 0.5293


## 7 · Evaluación en test y comparación con el baseline de 256

In [7]:
model.load_state_dict(torch.load('mejor_512.pt',map_location=DEVICE)); model.eval()
preds=[]; probs=[]; reales=[]
with torch.no_grad():
    for b in tt:
        ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE)
        with torch.cuda.amp.autocast(enabled=usar_amp):
            logits=model(ids,mask)
        pr=torch.softmax(logits.float(),dim=1).cpu().numpy()
        probs.extend(pr); preds.extend(pr.argmax(1)); reales.extend(b['labels'].numpy())
preds=np.array(preds); reales=np.array(reales); probs=np.array(probs)

f1m=f1_score(reales,preds,average='macro',zero_division=0)
print("="*58); print(f"RoBERTa clínico — max_length={MAX_LENGTH}  (TEST)"); print("="*58)
print(f"  F1-macro (7 clases): {f1m:.4f}   [baseline 256 = 0.5871]")
print(classification_report(reales,preds,target_names=[f'BI-RADS {i}' for i in range(7)],zero_division=0))

# Zona derivada: P(riesgo) = suma de prob de clases 4,5,6
p_riesgo=probs[:,4:7].sum(axis=1); y_zona=(reales>=4).astype(int)
auc=roc_auc_score(y_zona,p_riesgo)
yp=(p_riesgo>=0.5).astype(int); tn,fp,fn,tp=confusion_matrix(y_zona,yp,labels=[0,1]).ravel()
sens=tp/(tp+fn) if (tp+fn) else 0; espec=tn/(tn+fp) if (tn+fp) else 0
print("-"*58)
print(f"  AUC zona (derivado): {auc:.4f}   [baseline 256 = 0.93]")
print(f"  Sensibilidad: {sens:.3f} | Especificidad: {espec:.3f}")
print("="*58)
print("\n>>> ¿Mejoró? Compara F1 contra 0.5871 y AUC contra 0.93.")

/tmp/ipykernel_1658/2157636760.py:6: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=usar_amp):


RoBERTa clínico — max_length=512  (TEST)
  F1-macro (7 clases): 0.5431   [baseline 256 = 0.5871]
              precision    recall  f1-score   support

   BI-RADS 0       0.87      0.72      0.79       145
   BI-RADS 1       0.96      0.98      0.97        89
   BI-RADS 2       0.97      0.94      0.96       396
   BI-RADS 3       0.23      0.69      0.34        13
   BI-RADS 4       0.19      0.38      0.25         8
   BI-RADS 5       0.50      0.50      0.50         2
   BI-RADS 6       0.00      0.00      0.00         1

    accuracy                           0.88       654
   macro avg       0.53      0.60      0.54       654
weighted avg       0.92      0.88      0.90       654

----------------------------------------------------------
  AUC zona (derivado): 0.8835   [baseline 256 = 0.93]
  Sensibilidad: 0.636 | Especificidad: 0.981

>>> ¿Mejoró? Compara F1 contra 0.5871 y AUC contra 0.93.


## 8 · Cómo interpretar

- **Si el F1-macro sube de forma clara** (por ejemplo a 0,62+) y el AUC iguala o supera 0,93, entonces más contexto ayuda y vale la pena adoptar 512 (o 384) como configuración definitiva.
- **Si no cambia o baja**, confirma que la truncación a 256 no estaba perdiendo señal relevante, y conviene quedarse en 256 por costo computacional. Eso también es un resultado válido para documentar.
- Recuerda mirar la celda 4: el % de informes que superan 256 acota el efecto máximo posible.

**Si resulta bien:** descarga `mejor_512.pt` (panel de archivos de Colab) y registra el experimento en la rama `feat/512-tokens` del repositorio.